# wellcast quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/olaflaitinen/wellcast/blob/main/notebooks/01_quickstart.ipynb)

**Author:** Olaf Yunus Laitinen Imanov ([ORCID 0009-0006-5184-0810](https://orcid.org/0009-0006-5184-0810))  
**Affiliation:** Department of Mathematics and Statistics, University of Helsinki  
**License:** Mozilla Public License 2.0 (MPL-2.0), Copyright (c) 2026 Olaf Yunus Laitinen Imanov

A tour of the main workflow on a synthetic field: load data, check its quality, look at a well, fit a decline curve and flag anomalies. Everything runs on simulated data, so no files are needed.

In [ ]:
# Install wellcast when it is not available yet (for example on Google Colab).
import importlib.util

from IPython import get_ipython

if importlib.util.find_spec("wellcast") is None:
    get_ipython().run_line_magic(
        "pip",
        'install -q "wellcast[all]==0.1.0"',
    )

## 1. A synthetic field

`make_synthetic_field` simulates five producers and one water injector for five years of daily data. The physics is simple but consistent (Arps decline, IPR, choke equation, water breakthrough, shut-ins), and every anomaly it injects is labelled, so the analysis can be checked against the truth.

In [ ]:
import wellcast
from wellcast import plotting

field = wellcast.make_synthetic_field(n_wells=5, years=5, seed=42)
production = field.production
production.head()

In [ ]:
field.wells[["type", "qi", "di", "b", "bubble_point", "breakthrough_date"]]

## 2. Data quality

`qc_report` counts missing values, negative values, flatlines (frozen gauges) and physically impossible values per well and column. Columns a well never measures, such as the oil rate of an injector, show up as 100 percent missing.

In [ ]:
report = wellcast.qc_report(production)
report[report["missing"] < report["n"]].head(12)

## 3. Production history

Rates and pressures are drawn in two panels that share the time axis. Fluids keep fixed colors and dash patterns in every wellcast chart.

In [ ]:
plotting.production_history(production, "P-01")

## 4. Decline curve and EUR

`DeclineModel` fits exponential, harmonic and hyperbolic Arps models and keeps the one with the lowest AIC. A residual bootstrap gives P90 (low), P50 and P10 (high) estimates.

In [ ]:
rates = production.loc[production["well"] == "P-01", "oil_rate"]
model = wellcast.DeclineModel().fit(rates).bootstrap(200)
print(model)
model.eur_quantiles(econ_limit=20.0).round(0)

In [ ]:
model.plot(rates, econ_limit=20.0)

## 5. Anomalies

Physical rules and an Isolation Forest flag suspicious periods. The synthetic labels show what was injected.

In [ ]:
from wellcast.diagnostics import detect_anomalies

anomalies = detect_anomalies(production, method="rules")
anomalies.head(10)

In [ ]:
plotting.anomaly_overlay(production, anomalies, "bhp", "P-01")

## Next steps

* `02_decline_curve_analysis.ipynb`: decline models, model selection and uncertainty.
* `03_virtual_flow_metering.ipynb`: predicting bottomhole pressure with machine learning.
* `wellcast app` opens the interactive Streamlit app.